# Modelo de Indice de Eficiencia de Mobilidade

Este notebook cria um modelo simples para o servico `mobility-efficiency-index` usando duas fontes:
- `traffic-speed`
- `bus-operation-status`

Saidas do modelo:
- classe: `baixa`, `media`, `alta`
- indice: `efficiency_index` (0 a 100)


In [6]:
from pathlib import Path
import json
import re
import numpy as np
import pandas as pd
import joblib

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

TRAFFIC_PATH = Path('../../L0/traffic-speed/output-example.txt').resolve()
BUS_PATH = Path('../../L0/bus-operation-status/output-example.txt').resolve()
MODEL_PATH = Path('model.joblib').resolve()

TRAFFIC_PATH, BUS_PATH, MODEL_PATH


(PosixPath('/home/makleyston/Projects/service-composition/services/L0/traffic-speed/output-example.txt'),
 PosixPath('/home/makleyston/Projects/service-composition/services/L0/bus-operation-status/output-example.txt'),
 PosixPath('/home/makleyston/Projects/service-composition/services/L1/mobility-efficiency-index/model.joblib'))

In [7]:
def load_jsonl(path: Path) -> list[dict]:
    records = []
    with path.open('r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            records.append(json.loads(line))
    return records


def extract_id(text: str, pattern: str) -> float:
    if not text:
        return np.nan
    match = re.search(pattern, text)
    if not match:
        return np.nan
    try:
        return float(match.group(1))
    except ValueError:
        return np.nan


def parse_ts(ts: str):
    parsed = pd.to_datetime(ts, errors='coerce')
    if pd.isna(parsed):
        return np.nan, np.nan
    return float(parsed.hour), float(parsed.dayofweek)


def parse_traffic(records: list[dict]) -> pd.DataFrame:
    rows = []
    for rec in records:
        result = rec.get('saref:hasResult', {})
        ts = rec.get('saref:hasTimestamp')
        hour, weekday = parse_ts(ts)
        rows.append(
            {
                'traffic_status': result.get('saref:hasValue'),
                'traffic_confidence': result.get('urn:ufcity:confidence'),
                'traffic_segment_id': extract_id(rec.get('saref:hasFeatureOfInterest', ''), r'road-segment:(\d+)'),
                'traffic_hour': hour,
                'traffic_weekday': weekday,
            }
        )
    return pd.DataFrame(rows)


def parse_bus(records: list[dict]) -> pd.DataFrame:
    rows = []
    for rec in records:
        result = rec.get('saref:hasResult', {})
        ts = rec.get('saref:hasTimestamp')
        hour, weekday = parse_ts(ts)
        rows.append(
            {
                'bus_status': result.get('saref:hasValue'),
                'bus_confidence': result.get('urn:ufcity:confidence'),
                'bus_vehicle_id': extract_id(rec.get('saref:hasFeatureOfInterest', ''), r'bus:vehicle:(\d+)'),
                'bus_hour': hour,
                'bus_weekday': weekday,
            }
        )
    return pd.DataFrame(rows)


traffic_df = parse_traffic(load_jsonl(TRAFFIC_PATH))
bus_df = parse_bus(load_jsonl(BUS_PATH))

traffic_df.shape, bus_df.shape


((3681, 5), (3296, 5))

In [8]:
# Combinacao sintetica para PoC (nao existe alinhamento temporal real entre as fontes)
rng = np.random.default_rng(42)
sample_size = min(len(traffic_df), len(bus_df))

traffic_sample = traffic_df.sample(n=sample_size, random_state=42, replace=False).reset_index(drop=True)
bus_sample = bus_df.sample(n=sample_size, random_state=42, replace=False).reset_index(drop=True)

df = pd.concat([traffic_sample, bus_sample], axis=1)

numeric_cols = [
    'traffic_confidence', 'traffic_segment_id', 'traffic_hour', 'traffic_weekday',
    'bus_confidence', 'bus_vehicle_id', 'bus_hour', 'bus_weekday'
]
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

traffic_map = {'livre': 85, 'moderado': 55, 'congestionado': 20}
bus_map = {'normal': 85, 'lento': 55, 'parado': 20}

traffic_score = df['traffic_status'].map(traffic_map).fillna(50)
bus_score = df['bus_status'].map(bus_map).fillna(50)
avg_confidence = 50 + 50 * df[['traffic_confidence', 'bus_confidence']].mean(axis=1).fillna(0)
avg_confidence = np.clip(avg_confidence, 0, 100)

rule_score = 0.6 * traffic_score + 0.3 * bus_score + 0.1 * avg_confidence
df['efficiency_rule_score'] = np.clip(rule_score, 0, 100)

df['mobility_efficiency_class'] = pd.cut(
    df['efficiency_rule_score'],
    bins=[-1, 40, 70, 101],
    labels=['baixa', 'media', 'alta'],
).astype('object')

df['mobility_efficiency_class'].value_counts(normalize=True).rename('proporcao')


mobility_efficiency_class
media    0.780947
alta     0.180825
baixa    0.038228
Name: proporcao, dtype: float64

In [9]:
feature_cols = [
    'traffic_status', 'traffic_confidence', 'traffic_segment_id', 'traffic_hour', 'traffic_weekday',
    'bus_status', 'bus_confidence', 'bus_vehicle_id', 'bus_hour', 'bus_weekday'
]
target_col = 'mobility_efficiency_class'

X = df[feature_cols]
y = df[target_col]

numeric_features = [
    'traffic_confidence', 'traffic_segment_id', 'traffic_hour', 'traffic_weekday',
    'bus_confidence', 'bus_vehicle_id', 'bus_hour', 'bus_weekday'
]
categorical_features = ['traffic_status', 'bus_status']

numeric_transformer = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore')),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features),
    ]
)

clf = Pipeline(
    steps=[
        ('preprocessor', preprocessor),
        (
            'model',
            RandomForestClassifier(
                n_estimators=300,
                random_state=42,
                class_weight='balanced',
                n_jobs=-1,
            ),
        ),
    ]
)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

clf.fit(X_train, y_train)
pred = clf.predict(X_test)

print(classification_report(y_test, pred))
print(confusion_matrix(y_test, pred))


              precision    recall  f1-score   support

        alta       1.00      1.00      1.00       119
       baixa       1.00      1.00      1.00        25
       media       1.00      1.00      1.00       516

    accuracy                           1.00       660
   macro avg       1.00      1.00      1.00       660
weighted avg       1.00      1.00      1.00       660

[[119   0   0]
 [  0  25   0]
 [  0   0 516]]


In [10]:
index_weights = {'baixa': 0.0, 'media': 50.0, 'alta': 100.0}
classes = list(clf.named_steps['model'].classes_)
class_weight_vector = np.array([index_weights.get(c, 50.0) for c in classes], dtype=float)

proba = clf.predict_proba(X_test.head(10))
efficiency_index = np.clip(np.dot(proba, class_weight_vector), 0, 100)

preview = X_test.head(10).copy()
preview['eficiencia_real'] = y_test.head(10).values
preview['eficiencia_predita'] = clf.predict(X_test.head(10))
preview['efficiency_index'] = np.round(efficiency_index, 1)
preview


,traffic_status,traffic_confidence,traffic_segment_id,traffic_hour,traffic_weekday,bus_status,bus_confidence,bus_vehicle_id,bus_hour,bus_weekday,eficiencia_real,eficiencia_predita,efficiency_index
904,livre,0.660254,309.0,0.0,4.0,parado,1.000,20724.0,17.0,5.0,media,media,50.0
2318,livre,0.586344,374.0,0.0,4.0,normal,0.920,21252.0,17.0,5.0,alta,alta,99.7
1287,livre,0.539320,319.0,0.0,4.0,normal,0.670,30691.0,17.0,5.0,alta,alta,99.5
302,livre,0.871865,326.0,0.0,4.0,lento,0.930,30765.0,17.0,5.0,alta,alta,100.0
647,livre,0.856806,383.0,0.0,4.0,parado,0.620,11204.0,17.0,5.0,media,media,50.2
1194,congestionado,0.551820,370.0,0.0,4.0,parado,1.000,11285.0,17.0,5.0,baixa,baixa,0.3
3233,livre,0.545109,393.0,0.0,4.0,parado,0.965,11021.0,17.0,5.0,media,media,50.5
30,livre,0.615933,372.0,0.0,4.0,parado,1.000,11194.0,17.0,5.0,media,media,50.0
2271,livre,0.883814,309.0,0.0,4.0,parado,1.000,40738.0,17.0,5.0,media,media,50.0
415,livre,0.705923,319.0,0.0,4.0,parado,1.000,11067.0,17.0,5.0,media,media,49.8


In [11]:
artifact = {
    'pipeline': clf,
    'features': feature_cols,
    'target': target_col,
    'classes': classes,
    'index_name': 'efficiency_index',
    'index_scale': [0, 100],
    'index_weights': index_weights,
    'labeling_rule': {
        'score_formula': '0.6*traffic + 0.3*bus + 0.1*avg_confidence',
        'baixa': 'score <= 40',
        'media': '40 < score <= 70',
        'alta': 'score > 70',
    },
}

joblib.dump(artifact, MODEL_PATH)
MODEL_PATH


PosixPath('/home/makleyston/Projects/service-composition/services/L1/mobility-efficiency-index/model.joblib')